In [ ]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, os, random
import numpy as np
import pandas as pd
import warnings
import fastf1

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

cache_path = os.path.join(os.getcwd(), 'data', 'fastf1_cache')
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'fastf1  : {fastf1.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print(f'Cache   : {cache_path}')

import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

def fetch_results(seasons):
    rows = []
    for season in seasons:
        offset = 0
        while True:
            url = f'https://api.jolpi.ca/ergast/f1/{season}/results.json?limit=100&offset={offset}'
            resp = requests.get(url)
            data = resp.json()
            races = data['MRData']['RaceTable']['Races']
            if not races:
                break
            for race in races:
                for r in race['Results']:
                    pos_num = int(r['position']) if r['position'].isdigit() else None
                    status = r['status']
                    rows.append({
                        'season': int(race['season']),
                        'round': int(race['round']),
                        'race': race['raceName'],
                        'date': race['date'],
                        'circuit': race['Circuit']['circuitId'],
                        'driverId': r['Driver']['driverId'],
                        'driver': f"{r['Driver']['givenName']} {r['Driver']['familyName']}",
                        'constructor': r['Constructor']['name'],
                        'position': pos_num,
                        'positionText': r['positionText'],
                        'grid': int(r['grid']),
                        'laps': int(r['laps']),
                        'status': status,
                        'points': int(float(r['points'])),
                        'top10': pos_num is not None and pos_num <= 10,
                        'finished': status == 'Finished' or 'Lap' in status,
                    })
            total = int(data['MRData']['total'])
            offset += 100
            if offset >= total:
                break
    return pd.DataFrame(rows)

cache_file = os.path.join('data', 'processed', 'results_2022_2024.csv')
os.makedirs(os.path.dirname(cache_file), exist_ok=True)

if os.path.exists(cache_file):
    df = pd.read_csv(cache_file, parse_dates=['date'])
    df['top10'] = df['top10'].astype(bool)
    df['finished'] = df['finished'].astype(bool)
else:
    df = fetch_results([2022, 2023, 2024])
    df['date'] = pd.to_datetime(df['date'])
    df.to_csv(cache_file, index=False)

df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')
df['top10'] = df['top10'].astype(bool)

print(f'Total rows: {len(df)}')
print(f'Seasons: {sorted(df["season"].unique())}')
df.head()

# Decision-oriented EDA: 2022–2024 race results
All analyses follow: Question → Data → Answer → Decision.

## Question 1: Is the dataset balanced for the target (`Top-10`)?

### Data
We plot overall counts and season-wise proportions of `top10`.

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.countplot(x='top10', data=df, palette='Set2')
plt.title('Overall Top-10 vs Non-Top-10')
plt.xlabel('Top-10')
plt.ylabel('Count')
plt.subplot(1,2,2)
season_rate = df.groupby('season')['top10'].mean().reset_index()
sns.barplot(x='season', y='top10', data=season_rate, palette='Blues')
plt.title('Proportion of Top-10 by Season')
plt.ylabel('Proportion Top-10')
plt.xlabel('Season')
plt.tight_layout()

### Answer
Overall counts show roughly 10 finishers per race are Top-10 and about 10 are not (~50/50 at row level). Season-wise proportions are stable.
### Decision
Since the target is near-balanced, a majority-class baseline would only reach ~50% accuracy. We need proper metrics beyond accuracy alone.

## Question 2: Does starting grid position predict Top-10 finishes? (Trap check included)

### Data
Compare the distribution of `grid` between Top-10 and non-Top-10. Compute Spearman correlation. Explicitly check for survivorship bias.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='top10', y='grid', data=df, palette='Set3')
plt.title('Grid position by Top-10 (lower=better)')
plt.xlabel('Top-10')
plt.ylabel('Grid position')
plt.show()

corr, p = spearmanr(df['grid'].dropna(), df.loc[df['grid'].notna(),'top10'].astype(int))
print(f'Spearman correlation (all rows): {corr:.4f}, p={p:.2e}')

### Answer
Lower grid positions strongly associate with Top-10 finishes (Spearman ≈ −0.56, highly significant).

**Trap check (Survivorship bias):** Below we compare correlation on all rows vs only finishers. If we filter to `finished==True`, the correlation weakens, meaning DNFs are informative and removing them biases the analysis.
### Decision
Include `grid` as candidate feature. Handle DNFs explicitly to avoid survivorship bias.

In [ ]:
finished_df = df[df['finished']==True]
corr_all, _ = spearmanr(df['grid'].dropna(), df.loc[df['grid'].notna(),'top10'].astype(int))
corr_finished, _ = spearmanr(finished_df['grid'].dropna(), finished_df.loc[finished_df['grid'].notna(),'top10'].astype(int))
print(f'Spearman (all rows):      {corr_all:.4f}')
print(f'Spearman (finishers only): {corr_finished:.4f}')
print(f'Difference: {abs(corr_all) - abs(corr_finished):.4f} — filtering finishers weakens the signal')

## Question 3: Are temporal patterns stable across seasons? (Compare 2022 vs 2024)

### Data
Compare Top-10 rates and `grid` distributions between 2022 and 2024 to check for dataset shift.

In [ ]:
compare_seasons = df[df['season'].isin([2022,2024])]
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
sns.barplot(x='season', y='top10', data=compare_seasons.groupby('season')['top10'].mean().reset_index(), palette='rocket')
plt.title('Top-10 rate by season (2022 vs 2024)')
plt.ylabel('Proportion Top-10')
plt.subplot(1,2,2)
sns.boxplot(x='season', y='grid', data=compare_seasons, palette='pastel')
plt.title('Grid distribution by season')
plt.ylabel('Grid position')
plt.tight_layout()

### Answer
Top-10 rates are similar between 2022 and 2024. Grid distributions show minor shifts but nothing extreme. No large covariate shift detected.
### Decision
Temporal split is appropriate. Train on older seasons, validate on newer. Monitor drift if applying to future seasons.

## Question 4: Correlation analysis — which candidate features are associated with Top-10?

### Data
Compute Spearman correlations between `top10` and candidate features: `grid`, `position`, `points`, `laps`, `finished`.

In [ ]:
candidates = ['grid','position','points','laps','finished']
corrs = {}
for feat in candidates:
    series = df[feat].copy()
    if series.dtype == bool:
        series = series.astype(int)
    series = pd.to_numeric(series, errors='coerce')
    valid = df['top10'].notna() & series.notna()
    if valid.sum() > 0:
        s_corr, s_p = spearmanr(series[valid], df.loc[valid,'top10'].astype(int))
    else:
        s_corr, s_p = (np.nan, np.nan)
    corrs[feat] = (round(s_corr, 3), round(s_p, 4))
pd.DataFrame.from_dict(corrs, orient='index', columns=['spearman_r','p'])

### Answer
`grid` shows moderate negative correlation (better grid → more likely Top-10). `position` and `points` correlate very strongly but are **post-race** variables. `finished` correlates positively because DNFs rarely score Top-10.
### Decision
For pre-race prediction, only use `grid` (and potentially constructor/driver IDs). Exclude `position`, `points`, `status`, and `top10` as they are post-race.

## Question 5: Data quality audit (missingness, types, outliers)

### Data
Report missing values, data types, outliers, and classify missingness (MCAR/MAR/MNAR) for at least three columns.

In [ ]:
dq = pd.DataFrame(df.dtypes, columns=['dtype'])
dq['missing_count'] = df.isna().sum()
dq['missing_pct'] = (dq['missing_count']/len(df)).round(3)
print('=== Data quality summary ===')
print(dq.loc[['grid','status','points','position','top10']])
print(f'\nTotal rows: {len(df)}')
print(f'Grid=0 rows: {(df["grid"]==0).sum()}')
print(f'Unique status values: {df["status"].nunique()}')
print(f'Status categories: {sorted(df["status"].unique())}')

### Missingness classification
- `grid`: `grid=0` exists (pit lane starts) — possible MAR since it depends on qualifying events. No structural NaN.
- `position`: may have NaN for unclassified drivers — MNAR because missingness relates to the outcome (DNF).
- `points`: all drivers get at least 0 points, so no structural missingness. Complete.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['points'].dropna(), color='lightgreen')
plt.title('Points distribution (identify outliers)')
plt.xlabel('Points')
plt.show()

pre_race = ['grid','driverId','constructor','circuit']
post_race = ['position','points','status','top10']
print(f'Pre-race features:  {pre_race}')
print(f'Post-race features: {post_race}')

### Answer
Data quality is generally good. `points` has outliers at the high end (race winners score up to 26 pts). `grid=0` values exist and represent pit-lane starts. Pre-race vs post-race separation is clear.
### Decision
For prediction, only use pre-race features (`grid`, `constructor`, `driverId`, circuit). Handle `grid=0` carefully (impute or flag). Do not use post-race fields.

## Explicit temporal train / val / test split

In [ ]:
train_df = df[df['season'] == 2022].copy()
val_df = df[df['season'] == 2023].copy()
test_df = df[df['season'] == 2024].copy()

assert train_df['season'].max() < val_df['season'].min(), 'Leakage: train overlaps val'
assert val_df['season'].max() < test_df['season'].min(), 'Leakage: val overlaps test'
assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0, 'Empty split'

print(f'Train (2022): {len(train_df)} rows, dates {train_df["date"].min().date()} to {train_df["date"].max().date()}')
print(f'Val   (2023): {len(val_df)} rows, dates {val_df["date"].min().date()} to {val_df["date"].max().date()}')
print(f'Test  (2024): {len(test_df)} rows, dates {test_df["date"].min().date()} to {test_df["date"].max().date()}')
print('\nNo temporal leakage detected. All assertions passed.')

### Rationale
We use a chronological split by season to avoid information leakage. Training on 2022, validating on 2023, and testing on 2024 emulates the real scenario of predicting future races from past data. No random splits — temporal structure must be preserved.
### Decision
Train on 2022, tune/evaluate on 2023, hold out 2024 for final evaluation in Lab 2. Leakage verified programmatically via assertions above.

## Final 1-3-1 summary

**Headline (Decision):** Focus on qualifying — grid position is the strongest pre-race lever for a Top-10 finish.

**Evidence:**
- Grid position is the best pre-race predictor of Top-10 finishes (Spearman ≈ −0.56, p < 0.001).
- The Top-10 rate is stable across 2022–2024 (~50%), so models trained on older seasons should generalize.
- Filtering only finishers weakens the grid–result correlation, confirming survivorship bias must be avoided.

**Action:** Build models using grid position as the primary feature within a temporal split (Train=2022, Val=2023, Test=2024), and never use post-race variables to prevent leakage.